# 04 Scaled Model and Sampling

Embedding ve gizli katmanı büyütme, 2D embedding görselleştirme ve modelden isim örnekleme.


In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words, block_size=3):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

# 2D embedding modeli (cizim icin)
g = torch.Generator().manual_seed(2147483647)
C_2d = torch.randn((27, 2), generator=g, requires_grad=True)
W1_2d = torch.randn((6, 100), generator=g, requires_grad=True)
b1_2d = torch.randn(100, generator=g, requires_grad=True)
W2_2d = torch.randn((100, 27), generator=g, requires_grad=True)
b2_2d = torch.randn(27, generator=g, requires_grad=True)
params_2d = [C_2d, W1_2d, b1_2d, W2_2d, b2_2d]

for i in range(30000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C_2d[Xtr[ix]].view(-1, 6)
    h = torch.tanh(emb @ W1_2d + b1_2d)
    logits = h @ W2_2d + b2_2d
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in params_2d:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 20000 else 0.01
    for p in params_2d:
        p.data += -lr * p.grad

emb_dev = C_2d[Xdev].view(-1, 6)
loss_2d = F.cross_entropy(torch.tanh(emb_dev @ W1_2d + b1_2d) @ W2_2d + b2_2d, Ydev)
print("2D Embedding Dev Loss:", loss_2d.item())

# 2D Embedding Görselleştirme
plt.figure(figsize=(8, 8))
plt.scatter(C_2d[:, 0].data, C_2d[:, 1].data, s=250)
for i in range(C_2d.shape[0]):
    plt.text(C_2d[i, 0].item(), C_2d[i, 1].item(), itos[i], ha="center", va="center", color="white", fontsize=11)
plt.grid(True)
plt.title("2D Karakter Embedding Uzayi")
plt.show()

# Boyutlari buyutulmus model (emb_dim=10, hidden=200)
n_emb = 10
n_hidden = 200
C_sc = torch.randn((27, n_emb), generator=g, requires_grad=True)
W1_sc = torch.randn((n_emb * 3, n_hidden), generator=g, requires_grad=True)
b1_sc = torch.randn(n_hidden, generator=g, requires_grad=True)
W2_sc = torch.randn((n_hidden, 27), generator=g, requires_grad=True)
b2_sc = torch.randn(27, generator=g, requires_grad=True)
params_sc = [C_sc, W1_sc, b1_sc, W2_sc, b2_sc]

for i in range(35000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    emb = C_sc[Xtr[ix]].view(-1, n_emb * 3)
    h = torch.tanh(emb @ W1_sc + b1_sc)
    logits = h @ W2_sc + b2_sc
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in params_sc:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 25000 else 0.01
    for p in params_sc:
        p.data += -lr * p.grad

emb_dev = C_sc[Xdev].view(-1, n_emb * 3)
loss_sc = F.cross_entropy(torch.tanh(emb_dev @ W1_sc + b1_sc) @ W2_sc + b2_sc, Ydev)
print("Buyuk Model Dev Loss (10-dim, 200 hidden):", loss_sc.item())

# Modelden İsim Örnekleme (Sampling)
g_sample = torch.Generator().manual_seed(2147483647 + 10)
print("\n--- MLP Uretilen Isimler ---")
for _ in range(10):
    out = []
    context = [0] * 3
    while True:
        emb = C_sc[torch.tensor([context])].view(1, -1)
        h = torch.tanh(emb @ W1_sc + b1_sc)
        logits = h @ W2_sc + b2_sc
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print("".join(out))
